# Day 2 — Data Cleaning + SQLite Load
## Bluestock MF Capstone

**Day 1 built:** ETL pipeline, initial 19-fund DB, csv_ingestion_adapter  
**Day 2 adds:** All 40 real funds, star schema, 10 cleaned CSVs, 10 SQL queries

In [1]:
from pathlib import Path
import pandas as pd, sqlite3, warnings
warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
DB   = BASE / "data" / "db" / "bluestock_mf.db"
PROC = BASE / "data" / "processed"
print("DB exists:", DB.exists())
print("Processed dir:", PROC.exists())


DB exists: True
Processed dir: True


In [2]:
# Cleaned CSV Inventory
import os
print(f"{'File':45s} {'Rows':>8}  {'Cols':>5}")
print("-" * 65)
for f in sorted(PROC.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name:43s} {len(df):>8,}  {len(df.columns):>5}")


File                                              Rows   Cols
-----------------------------------------------------------------
  01_fund_master_clean.csv                          40     15
  02_nav_history_clean.csv                      46,000      5
  03_aum_by_fund_house_clean.csv                    90      5
  04_monthly_sip_inflows_clean.csv                  48      7
  05_category_inflows_clean.csv                    144      3
  06_industry_folio_count_clean.csv                 21      8
  07_scheme_performance_clean.csv                   40     21
  08_investor_transactions_clean.csv            32,778     13
  09_portfolio_holdings_clean.csv                  322      8
  10_benchmark_indices_clean.csv                 8,050      4
  alpha_beta.csv                                    40      8
  fund_scorecard.csv                                40     19


In [3]:
# Star Schema Row Count Validation
tables = [("dim_fund",40),("dim_date",1297),("fact_nav",46000),
          ("fact_transactions",32778),("fact_performance",40),("fact_aum",90),
          ("ref_sip_inflows",48),("ref_category_inflows",144),
          ("ref_folio_count",21),("ref_portfolio_holdings",322),
          ("ref_benchmark_indices",8050)]

with sqlite3.connect(DB) as conn:
    print(f"{'Table':30s} {'Actual':>10}  {'Expected':>10}  Status")
    print("-" * 65)
    for table, expected in tables:
        actual = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        ok = "OK" if actual >= expected else "WARN"
        print(f"  {table:28s} {actual:>10,}  {expected:>10,}  {ok}")


Table                              Actual    Expected  Status
-----------------------------------------------------------------
  dim_fund                             40          40  OK
  dim_date                          1,297       1,297  OK
  fact_nav                         46,000      46,000  OK
  fact_transactions                32,778      32,778  OK
  fact_performance                     40          40  OK
  fact_aum                             90          90  OK
  ref_sip_inflows                      48          48  OK
  ref_category_inflows                144         144  OK
  ref_folio_count                      21          21  OK
  ref_portfolio_holdings              322         322  OK
  ref_benchmark_indices             8,050       8,050  OK


In [4]:
# NAV History Quality Checks
nav = pd.read_csv(PROC / "02_nav_history_clean.csv", parse_dates=["date"])
print("NAV HISTORY QUALITY:")
print(f"  Rows            : {len(nav):,}")
print(f"  Funds           : {nav.scheme_code.nunique()}")
print(f"  Date range      : {nav.date.min().date()} to {nav.date.max().date()}")
print(f"  Weekend dates   : {(nav.date.dt.dayofweek >= 5).sum()}")
print(f"  NAV <= 0        : {(nav.nav <= 0).sum()}")
print(f"  Duplicates      : {nav.duplicated(['scheme_code','date']).sum()}")
print(f"  Null NAV        : {nav.nav.isna().sum()}")


NAV HISTORY QUALITY:
  Rows            : 46,000
  Funds           : 40
  Date range      : 2022-01-03 to 2026-05-29
  Weekend dates   : 0
  NAV <= 0        : 0
  Duplicates      : 0
  Null NAV        : 0


In [5]:
# Transaction Quality Checks
txn = pd.read_csv(PROC / "08_investor_transactions_clean.csv")
print("TRANSACTION QUALITY:")
print(f"  Rows            : {len(txn):,}")
print(f"  Types           : {sorted(txn.transaction_type.unique())}")
print(f"  KYC status      : {sorted(txn.kyc_status.unique())}")
print(f"  City tiers      : {sorted(txn.city_tier.unique())}")
print(f"  Amount <= 0     : {(txn.amount_inr <= 0).sum()}")
print(f"  States          : {txn.state.nunique()}")
print()
print(txn.transaction_type.value_counts().to_string())


TRANSACTION QUALITY:
  Rows            : 32,778
  Types           : ['Lumpsum', 'Redemption', 'SIP']
  KYC status      : ['Pending', 'Verified']
  City tiers      : ['B30', 'T30']
  Amount <= 0     : 0
  States          : 12

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967


In [6]:
# Scheme Performance Quality Checks
sch = pd.read_csv(PROC / "07_scheme_performance_clean.csv")
print("SCHEME PERFORMANCE QUALITY:")
print(f"  Rows            : {len(sch)}")
print(f"  Anomaly flags   : {sch.anomaly_flag.sum()}")
print(f"  ER out of range : {(~sch.er_valid).sum()}")
print()
print(sch[['return_1yr_pct','return_3yr_pct','return_5yr_pct','expense_ratio_pct']].describe().round(2))


SCHEME PERFORMANCE QUALITY:
  Rows            : 40
  Anomaly flags   : 0
  ER out of range : 0

       return_1yr_pct  return_3yr_pct  return_5yr_pct  expense_ratio_pct
count           40.00           40.00           40.00              40.00
mean            14.38           14.09           14.52               1.24
std              4.88            4.62            4.45               0.39
min              4.26            5.14            5.43               0.55
25%             11.74           12.04           12.34               0.79
50%             14.62           14.20           14.18               1.42
75%             16.39           15.88           17.58               1.54
max             24.93           23.39           23.80               1.64


In [7]:
# Q1 Top 5 by AUM | Q2 Avg NAV per Month
with sqlite3.connect(DB) as conn:
    print("Q1: TOP 5 FUNDS BY AUM")
    q1 = pd.read_sql_query(
        "SELECT f.scheme_name, f.amc_name, p.aum_crore, p.return_3yr_pct "
        "FROM fact_performance p JOIN dim_fund f ON p.scheme_code=f.scheme_code "
        "ORDER BY p.aum_crore DESC LIMIT 5", conn)
    print(q1.to_string(index=False))
    print()
    print("Q2: AVG NAV PER MONTH (first 8)")
    q2 = pd.read_sql_query(
        "SELECT d.year, d.month_name, ROUND(AVG(n.nav),2) avg_nav "
        "FROM fact_nav n JOIN dim_date d ON n.date_key=d.date_key "
        "GROUP BY d.year, d.month ORDER BY d.year, d.month LIMIT 8", conn)
    print(q2.to_string(index=False))


Q1: TOP 5 FUNDS BY AUM
                                          scheme_name          amc_name  aum_crore  return_3yr_pct
Mirae Asset Emerging Bluechip Fund - Regular - Growth    Mirae Asset MF      49046           14.56
        Kotak Emerging Equity Fund - Regular - Growth Kotak Mahindra MF      47469           18.23
       Nippon India Small Cap Fund - Regular - Growth   Nippon India MF      43630           20.15
           DSP Top 100 Equity Fund - Regular - Growth   DSP Mutual Fund      41828           12.82
                  UTI Mid Cap Fund - Regular - Growth   UTI Mutual Fund      41728           15.61

Q2: AVG NAV PER MONTH (first 8)
 year month_name  avg_nav
 2022        Jan   207.06
 2022        Feb   207.72
 2022        Mar   209.69
 2022        Apr   211.83
 2022        May   212.73
 2022        Jun   213.86
 2022        Jul   213.96
 2022        Aug   215.68


In [8]:
# Q3 SIP YoY | Q4 Transactions by State | Q5 Low expense ratio funds
with sqlite3.connect(DB) as conn:
    print("Q3: SIP YoY GROWTH")
    q3 = pd.read_sql_query(
        "SELECT strftime('%Y',month) yr, ROUND(SUM(sip_inflow_crore),0) total_sip, "
        "ROUND(AVG(yoy_growth_pct),2) avg_yoy FROM ref_sip_inflows GROUP BY yr ORDER BY yr", conn)
    print(q3.to_string(index=False))
    print()
    print("Q4: TRANSACTIONS BY STATE (top 8)")
    q4 = pd.read_sql_query(
        "SELECT state, COUNT(*) txns, ROUND(SUM(amount_inr)/1e7,1) crore "
        "FROM fact_transactions GROUP BY state ORDER BY crore DESC LIMIT 8", conn)
    print(q4.to_string(index=False))
    print()
    print("Q5: FUNDS WITH EXPENSE RATIO < 1%")
    q5 = pd.read_sql_query(
        "SELECT f.scheme_name, p.expense_ratio_pct, p.return_3yr_pct, p.sharpe_ratio "
        "FROM fact_performance p JOIN dim_fund f ON p.scheme_code=f.scheme_code "
        "WHERE p.expense_ratio_pct < 1.0 ORDER BY p.expense_ratio_pct", conn)
    print(q5.to_string(index=False))


Q3: SIP YoY GROWTH
  yr  total_sip  avg_yoy
2022   149437.0      NaN
2023   184763.0    23.49
2024   269781.0    45.69
2025   335740.0    25.19

Q4: TRANSACTIONS BY STATE (top 8)
         state  txns  crore
        Punjab  2965   31.6
    Tamil Nadu  2806   31.5
Madhya Pradesh  2931   30.8
     Rajasthan  2577   29.9
       Gujarat  2780   29.8
   West Bengal  2748   29.7
     Telangana  2718   29.0
         Delhi  2677   29.0

Q5: FUNDS WITH EXPENSE RATIO < 1%
                                         scheme_name  expense_ratio_pct  return_3yr_pct  sharpe_ratio
Nippon India Gilt Securities Fund - Regular - Growth               0.55            5.31          1.33
        HDFC Short Term Debt Fund - Regular - Growth               0.56            7.37          1.84
                Kotak Liquid Fund - Regular - Growth               0.60            6.18          6.18
            SBI Bluechip Fund - Direct Plan - Growth               0.66           11.30          0.81
           SBI Small Cap

In [9]:
# Q6–Q10
with sqlite3.connect(DB) as conn:
    print("Q6: CATEGORY NET INFLOWS")
    q6 = pd.read_sql_query(
        "SELECT category, ROUND(SUM(net_inflow_crore),0) net_inflow "
        "FROM ref_category_inflows GROUP BY category ORDER BY net_inflow DESC", conn)
    print(q6.to_string(index=False))
    print()
    print("Q7: FUND HOUSE AUM MARKET SHARE 2025")
    q7 = pd.read_sql_query(
        "SELECT fund_house, ROUND(SUM(aum_lakh_crore),2) aum_lc, "
        "ROUND(SUM(aum_lakh_crore)/SUM(SUM(aum_lakh_crore)) OVER()*100,1) pct "
        "FROM fact_aum WHERE date_key LIKE '2025%' GROUP BY fund_house ORDER BY aum_lc DESC", conn)
    print(q7.to_string(index=False))
    print()
    print("Q8: TOP 10 FUNDS BY 3Y CAGR")
    q8 = pd.read_sql_query(
        "SELECT f.scheme_name, p.return_3yr_pct, p.sharpe_ratio, p.max_drawdown_pct "
        "FROM fact_performance p JOIN dim_fund f ON p.scheme_code=f.scheme_code "
        "ORDER BY p.return_3yr_pct DESC LIMIT 10", conn)
    print(q8.to_string(index=False))


Q6: CATEGORY NET INFLOWS
         category  net_inflow
           Liquid    451275.0
Sectoral/Thematic    103829.0
        Flexi Cap     63989.0
  Large & Mid Cap     57752.0
   Short Duration     55530.0
          Mid Cap     55312.0
        Small Cap     46596.0
           Hybrid     38868.0
        Large Cap     25633.0
     Value/Contra     16980.0
             Gilt     10395.0
             ELSS      6080.0

Q7: FUND HOUSE AUM MARKET SHARE 2025
              fund_house  aum_lc  pct
         SBI Mutual Fund   25.00 21.3
     ICICI Prudential MF   19.54 16.7
        HDFC Mutual Fund   17.25 14.7
         Nippon India MF   12.60 10.7
       Kotak Mahindra MF   10.72  9.1
Aditya Birla Sun Life MF    8.45  7.2
         UTI Mutual Fund    7.65  6.5
        Axis Mutual Fund    6.60  5.6
          Mirae Asset MF    5.15  4.4
         DSP Mutual Fund    4.25  3.6

Q8: TOP 10 FUNDS BY 3Y CAGR
                                       scheme_name  return_3yr_pct  sharpe_ratio  max_drawdown_pct
 

In [10]:
with sqlite3.connect(DB) as conn:
    print("Q9: SIP DEMOGRAPHICS BY AGE+GENDER")
    q9 = pd.read_sql_query(
        "SELECT age_group, gender, COUNT(*) sip_txns, ROUND(AVG(amount_inr),0) avg_amount "
        "FROM fact_transactions WHERE transaction_type='SIP' "
        "GROUP BY age_group, gender ORDER BY age_group, gender", conn)
    print(q9.to_string(index=False))
    print()
    print("Q10: FOLIO COUNT GROWTH BY QUARTER")
    q10 = pd.read_sql_query(
        "SELECT strftime('%Y',month) yr, "
        "CASE WHEN CAST(strftime('%m',month) AS INT)<=3 THEN 'Q1' "
        "     WHEN CAST(strftime('%m',month) AS INT)<=6 THEN 'Q2' "
        "     WHEN CAST(strftime('%m',month) AS INT)<=9 THEN 'Q3' ELSE 'Q4' END qtr, "
        "MAX(total_folios_crore) end_folios, MIN(total_folios_crore) start_folios "
        "FROM ref_folio_count GROUP BY yr, qtr ORDER BY yr, qtr", conn)
    print(q10.to_string(index=False))

print()
print("Day 2 Complete — 10 cleaned CSVs, star schema loaded, 10 SQL queries validated")


Q9: SIP DEMOGRAPHICS BY AGE+GENDER
age_group gender  sip_txns  avg_amount
    18-25 Female      1041     10969.0
    18-25   Male      1908     10944.0
    26-35 Female      2647     10948.0
    26-35   Male      5416     11006.0
    36-45 Female      1619     10778.0
    36-45   Male      3307     10938.0
    46-55 Female       757     10898.0
    46-55   Male      1543     11254.0
      56+ Female       530     10691.0
      56+   Male       948     12069.0

Q10: FOLIO COUNT GROWTH BY QUARTER
  yr qtr  end_folios  start_folios
2022  Q1       13.26         13.26
2022  Q2       13.91         13.91
2022  Q3       13.85         13.85
2022  Q4       14.12         14.12
2023  Q1       14.81         14.81
2023  Q2       15.54         15.54
2023  Q3       16.28         16.28
2023  Q4       16.72         16.72
2024  Q1       17.78         17.78
2024  Q2       18.85         18.85
2024  Q3       19.98         19.98
2024  Q4       22.89         21.62
2025  Q1       23.45         23.10
2025  Q2  